# LeafDoctor: train on Kaggle

Thin wrapper around `src/train.py`; all logic lives in the repo.

Before running:
1. **Add Data** → attach the private `ishaanbanga/leafdoctor-data` dataset (made with `scripts/pack_for_kaggle.py`).
   It is mounted read-only at `/kaggle/input/datasets/ishaanbanga/leafdoctor-data/`.
2. **Settings** → Accelerator: GPU, Internet: on (to clone the repo and download ImageNet weights).
3. Set the parameters below, then **Run All** (or *Save Version* to run in the background).

All outputs go to `/kaggle/working/results/<arm>_seed<seed>/` (outside the cloned repo, so re-running
the clone cell never deletes earlier runs).

In [ ]:
# Parameters
REPO_URL = "https://github.com/bangaishaan/leafdoctor.git"
BRANCH = "main"
ARM = "standard"          # standard | cutpaste | diffusion
SEED = 1
EPOCHS = 15               # total, including train.py's 3 linear-probe epochs (--lp-epochs)
EXTRA_TRAIN_DIR = ""      # required for cutpaste/diffusion, e.g. "/kaggle/input/datasets/ishaanbanga/leafdoctor-cutpaste"
DATA_ROOT = "/kaggle/input/datasets/ishaanbanga/leafdoctor-data"   # read-only
OUT_DIR = "/kaggle/working/results"                                # all outputs

In [ ]:
%cd /kaggle/working
!rm -rf leafdoctor && git clone --depth 1 --branch {BRANCH} {REPO_URL} leafdoctor
%cd /kaggle/working/leafdoctor
!git log -1 --oneline
!pip install -q -r requirements.txt

In [ ]:
import os
for name in ("plantvillage", "plantdoc", "class_map.csv", "split_manifest.csv"):
    assert os.path.exists(os.path.join(DATA_ROOT, name)), f"{name} not found in {DATA_ROOT}; attach the dataset (Add Data)"
os.makedirs(OUT_DIR, exist_ok=True)
print("data root:", DATA_ROOT, "| outputs:", OUT_DIR)
extra = f'--extra-train-dir "{EXTRA_TRAIN_DIR}"' if EXTRA_TRAIN_DIR else ""

In [ ]:
!python src/train.py --data-root "{DATA_ROOT}" --out "{OUT_DIR}" --arm {ARM} --seed {SEED} --epochs {EPOCHS} {extra}

In [ ]:
!ls -la "{OUT_DIR}/{ARM}_seed{SEED}"

In [ ]:
# Print results to the log (so they survive even if Kaggle's output saving fails), then shrink /kaggle/working
import shutil
from pathlib import Path

run_dir = Path(OUT_DIR) / f"{ARM}_seed{SEED}"
for name in ("history.csv", "metrics.json"):
    f = run_dir / name
    print(f"===== {f} =====")
    print(f.read_text(encoding="utf-8") if f.exists() else "(missing: check the training log above)", flush=True)

# Leave the clone before deleting it; outputs live in OUT_DIR, outside the clone.
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/leafdoctor", ignore_errors=True)
print("deleted /kaggle/working/leafdoctor")

# Keep each run's best.pt, delete any other checkpoint.
for ckpt in sorted(p for ext in ("*.pt", "*.pth") for p in Path(OUT_DIR).rglob(ext)):
    if ckpt.name != "best.pt":
        ckpt.unlink()
        print("deleted", ckpt)
!du -sh /kaggle/working